# Projet #3 — Prédiction du Churn Client
## Notebook 2 — Modélisation, optimisation et validation

Ce notebook répond aux **deuxième et troisième livrables** du brief :
- mettre en œuvre et justifier le processus d'apprentissage ;
- comparer Régression Logistique, Arbre de Décision et Random Forest ;
- optimiser les hyperparamètres par validation croisée / recherche en grille ;
- mesurer Accuracy, Precision, Recall, F1 et ROC-AUC ;
- analyser la capacité de généralisation sur un jeu de test tenu à l'écart.

### Choix méthodologique
La **régression logistique** sert de modèle de référence : elle est adaptée à une cible binaire, rapide et interprétable. L'arbre de décision apporte des règles explicables. La Random Forest permet de réduire la variance d'un arbre unique et de modéliser des relations non linéaires.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/processed/telco_churn_clean.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Exécutez d'abord 01_exploration_preparation.ipynb pour créer "
        "data/processed/telco_churn_clean.csv"
    )

df = pd.read_csv(DATA_PATH)
df.shape

## 1. Séparation des données et prévention des fuites

Le jeu de test (20 %) est isolé **avant** l'apprentissage. La stratification conserve la proportion de churn. Toutes les transformations sont apprises uniquement sur le jeu d'entraînement via un `Pipeline`.


In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Churn train :", round(y_train.mean(), 4), "| Churn test :", round(y_test.mean(), 4))

## 2. Modèles de référence

On commence avec des hyperparamètres raisonnables sans optimisation. `class_weight="balanced"` est utilisé afin d'accorder davantage d'importance à la classe minoritaire (clients qui churnent), ce qui est cohérent avec le besoin métier de détecter les clients à risque.


In [ ]:
baseline_models = {
    "Régression logistique": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Arbre de décision": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
    ),
}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    metrics = {
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    return pipe, metrics

baseline_fitted = {}
baseline_results = []

for name, model in baseline_models.items():
    fitted, metrics = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    baseline_fitted[name] = fitted
    baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results).set_index("Modèle").round(4)
baseline_df

### Pourquoi plusieurs métriques ?

- **Accuracy** : proportion globale de bonnes prédictions.
- **Precision** : parmi les clients signalés à risque, proportion réellement churn.
- **Recall** : parmi les clients qui churnent réellement, proportion détectée.
- **F1** : compromis entre precision et recall.
- **ROC-AUC** : capacité du modèle à classer correctement les clients à risque sur l'ensemble des seuils.

Dans un contexte de rétention, le **recall** est particulièrement important : un faux négatif correspond à un client à risque que l'entreprise ne détecte pas.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, pipe) in zip(axes, baseline_fitted.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
ax = plt.gca()
for name, pipe in baseline_fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.title("Courbes ROC — modèles de référence")
plt.tight_layout()
plt.show()

## 3. Validation croisée

Avant l'optimisation, une validation croisée stratifiée à 5 plis donne une estimation plus robuste des performances sur les données d'entraînement.


In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_rows = []
for name, model in baseline_models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Modèle": name,
        **{metric: scores[f"test_{metric}"].mean() for metric in scoring}
    })

cv_df = pd.DataFrame(cv_rows).set_index("Modèle").round(4)
cv_df

## 4. Optimisation des hyperparamètres avec GridSearchCV

Le critère principal retenu est le **ROC-AUC**, car il mesure la capacité de discrimination indépendamment d'un seuil fixe. Le recall reste suivi séparément pour la décision métier.


In [ ]:
search_spaces = {
    "Régression logistique": (
        LogisticRegression(max_iter=2500, class_weight="balanced", random_state=42),
        {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["liblinear", "lbfgs"],
        }
    ),
    "Arbre de décision": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {
            "model__max_depth": [3, 5, 7, 10, None],
            "model__min_samples_leaf": [5, 10, 20, 40],
            "model__criterion": ["gini", "entropy"],
        }
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        {
            "model__n_estimators": [200, 400],
            "model__max_depth": [None, 8, 14],
            "model__min_samples_leaf": [1, 3, 8],
            "model__max_features": ["sqrt", "log2"],
        }
    ),
}

best_models = {}
search_summary = []

for name, (model, grid) in search_spaces.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    search = GridSearchCV(
        pipe,
        param_grid=grid,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    search_summary.append({
        "Modèle": name,
        "Meilleur CV ROC-AUC": search.best_score_,
        "Meilleurs paramètres": search.best_params_
    })

pd.DataFrame(search_summary).set_index("Modèle")

## 5. Évaluation finale sur le jeu de test

Le jeu de test n'a pas servi à la recherche d'hyperparamètres. Cette étape mesure donc la capacité de généralisation des modèles optimisés.


In [ ]:
final_rows = []
for name, pipe in best_models.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    final_rows.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })

final_df = pd.DataFrame(final_rows).set_index("Modèle").sort_values("ROC-AUC", ascending=False)
final_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, pipe) in zip(axes, best_models.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} optimisé")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
ax = plt.gca()
for name, pipe in best_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.title("Courbes ROC — modèles optimisés")
plt.tight_layout()
plt.show()

## 6. Interprétation des variables

Pour la Random Forest, l'importance des variables permet d'identifier les caractéristiques les plus utilisées par l'ensemble des arbres. Pour la régression logistique, le signe et l'amplitude des coefficients renseignent sur la direction de l'association avec le churn (après transformation).


In [ ]:
rf_pipe = best_models["Random Forest"]
feature_names = rf_pipe.named_steps["prep"].get_feature_names_out()
rf_importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = (pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importances
})
.sort_values("importance", ascending=False)
.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, y="feature", x="importance")
plt.title("Top 20 — importance des caractéristiques (Random Forest)")
plt.tight_layout()
plt.show()

importance_df

In [ ]:
log_pipe = best_models["Régression logistique"]
log_features = log_pipe.named_steps["prep"].get_feature_names_out()
coefs = log_pipe.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": log_features,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
}).sort_values("abs_coefficient", ascending=False).head(20)

coef_df[["feature", "coefficient"]]

## 7. Analyse de la généralisation

On compare le ROC-AUC moyen obtenu en validation croisée avec le ROC-AUC du jeu de test. Un écart faible suggère une bonne stabilité. Un score d'entraînement/CV très supérieur au test serait un signe de surapprentissage.


In [ ]:
generalization = []
for row in search_summary:
    name = row["Modèle"]
    cv_auc = row["Meilleur CV ROC-AUC"]
    test_auc = float(final_df.loc[name, "ROC-AUC"])
    generalization.append({
        "Modèle": name,
        "CV ROC-AUC": cv_auc,
        "Test ROC-AUC": test_auc,
        "Écart absolu": abs(cv_auc - test_auc),
    })

pd.DataFrame(generalization).set_index("Modèle").round(4)

## 8. Conclusion et recommandation métier

Le meilleur modèle ne doit pas être choisi uniquement sur l'accuracy. Dans un cas de churn, l'entreprise cherche surtout à identifier suffisamment tôt les clients susceptibles de partir.

La décision finale peut donc privilégier :
- le **ROC-AUC** pour la qualité globale de classement ;
- le **recall** si le coût d'un client perdu est supérieur au coût d'une action de rétention inutile ;
- la **precision** si les campagnes de rétention sont coûteuses et doivent cibler moins de clients.

La régression logistique reste particulièrement intéressante lorsqu'elle offre des performances proches des modèles d'ensemble, car elle est plus simple à expliquer aux équipes métier. La Random Forest peut être préférée si son gain de performance est significatif et si l'explicabilité par importance des variables est suffisante.

### Limites
- Le dataset est historique et relativement petit.
- Aucune information de coût réel de churn / campagne de rétention n'est fournie ; le seuil de décision 0,5 n'est donc pas forcément optimal.
- Les associations observées ne prouvent pas une causalité.
- En production, il faudrait surveiller la dérive des données et des performances.

### Pistes de déploiement
Le modèle retenu pourrait être sérialisé (`joblib`) et exposé via FastAPI, avec un tableau de bord de suivi. Une CI pourrait vérifier les tests et le pipeline, tandis qu'un outil de monitoring (par exemple Evidently) suivrait les distributions et les performances en production.
